# 03 — Evaluation & Error Analysis

Deep-dive into model performance on the test set:
- Clarke Error Grid Analysis (clinical safety metric)
- MARD at 30-min and 60-min horizons
- Failure case analysis — where does the model get it wrong?
- Time-in-Range impact of prediction errors

> The Clarke Error Grid was chosen as the primary metric because it directly maps prediction
> errors to clinical risk, not just numerical accuracy. A model that always predicts 130 mg/dL
> may have low RMSE but fail catastrophically during hypoglycemia.

In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from data.sample.generate_synthetic import simulate_cgm
from src.models import GlucoseLSTM
from src.preprocessing import CGMProcessor
from src.evaluation import (
    mard, rmse, mae, clarke_error_grid, plot_clarke_error_grid
)
from src.visualization import plot_glucose_trace

plt.rcParams['figure.dpi'] = 120

## 1. Load model and generate predictions

This notebook assumes you have run notebook 02 or `train.py` to produce a saved checkpoint.
If not, it initialises an untrained LSTM and you'll see random predictions — train first!

In [ ]:
from pathlib import Path

df_raw = simulate_cgm(days=30, seed=42)
processor = CGMProcessor(seq_len=24, pred_horizons=[6, 12])
df = processor.clean(df_raw)

n = len(df)
test_df = df.iloc[int(n * 0.85):]
processor.fit_scaler(df.iloc[:int(n * 0.7)])
X_test, y_test = processor.make_windows(test_df)

model = GlucoseLSTM(hidden_size=64, num_layers=2, num_horizons=2)
ckpt = Path('../models/checkpoints/lstm_best.pth')
if ckpt.exists():
    model.load_state_dict(torch.load(ckpt, map_location='cpu'))
    print('Loaded trained checkpoint.')
else:
    print('No checkpoint found — using random weights. Run train.py first.')

model.eval()
with torch.no_grad():
    preds_norm = model(torch.tensor(X_test, dtype=torch.float32)).numpy()

true_mg = processor.inverse_transform(y_test)
pred_mg = processor.inverse_transform(preds_norm)

print(f'Test samples: {len(true_mg)}')

## 2. Clarke Error Grid Analysis — 30-min horizon

In [ ]:
zones_30, zone_idx_30 = clarke_error_grid(true_mg[:, 0], pred_mg[:, 0])
print('Clarke EGA zones (30-min horizon):')
for z, pct in zones_30.items():
    flag = ' ← GOOD' if z == 'A' else (' ← DANGEROUS' if z == 'E' else '')
    print(f'  Zone {z}: {pct:.1f}%{flag}')

fig = plot_clarke_error_grid(
    true_mg[:, 0], pred_mg[:, 0],
    title='Clarke Error Grid — LSTM, 30-min horizon',
    save_path='../results/plots/03_ceg_30min.png',
)
plt.show()

## 3. Clarke EGA — 60-min horizon

In [ ]:
zones_60, _ = clarke_error_grid(true_mg[:, 1], pred_mg[:, 1])
print('Clarke EGA zones (60-min horizon):')
for z, pct in zones_60.items():
    print(f'  Zone {z}: {pct:.1f}%')

fig = plot_clarke_error_grid(
    true_mg[:, 1], pred_mg[:, 1],
    title='Clarke Error Grid — LSTM, 60-min horizon',
    save_path='../results/plots/03_ceg_60min.png',
)
plt.show()

## 4. MARD and RMSE summary table

In [ ]:
rows = []
for i, (h_min, label) in enumerate([(30, '30-min'), (60, '60-min')]):
    t, p = true_mg[:, i], pred_mg[:, i]
    rows.append({'Horizon': label,
                 'MARD (%)': round(mard(t, p), 2),
                 'RMSE (mg/dL)': round(rmse(t, p), 2),
                 'MAE (mg/dL)': round(mae(t, p), 2),
                 'Zone A (%)': round(list(clarke_error_grid(t, p)[0].values())[0], 1)})

results_df = pd.DataFrame(rows).set_index('Horizon')
print(results_df.to_string())
results_df.to_csv('../results/metrics/lstm_results.csv')

## 5. Failure case analysis

We look specifically at the predictions the model got most wrong (Zone D/E) and ask: **why?**
Typical patterns to look for:
- Rapid postprandial rise the model didn't anticipate
- Nocturnal hypoglycemia during a flat-looking window
- High glycemic variability confusing the trend signal

In [ ]:
_, zone_idx_30 = clarke_error_grid(true_mg[:, 0], pred_mg[:, 0])
bad_mask = zone_idx_30['D'] | zone_idx_30['E'] | zone_idx_30['C']
bad_indices = np.where(bad_mask)[0]

print(f'Problematic predictions (zones C/D/E): {bad_mask.sum()} / {len(true_mg)} '
      f'({bad_mask.mean()*100:.1f}%)')

if len(bad_indices) > 0:
    n_show = min(4, len(bad_indices))
    fig, axes = plt.subplots(n_show, 1, figsize=(12, 3.5 * n_show))
    if n_show == 1: axes = [axes]

    for ax, idx in zip(axes, bad_indices[:n_show]):
        window_mg = processor.inverse_transform(X_test[idx, :, 0].reshape(1, -1)).flatten()
        true_val  = true_mg[idx, 0]
        pred_val  = pred_mg[idx, 0]
        error_pct = abs(true_val - pred_val) / true_val * 100

        ax.plot(range(24), window_mg, 'o-', color='#60a5fa', ms=3, label='Input window')
        ax.scatter([30], [true_val], color='#4ade80', s=80, zorder=5, label=f'True: {true_val:.0f}')
        ax.scatter([30], [pred_val], color='#ef4444', marker='x', s=120, zorder=5,
                   label=f'Pred: {pred_val:.0f} (err: {error_pct:.0f}%)')
        ax.axhspan(70, 180, alpha=0.07, color='#4ade80')
        ax.set_title(f'Failure case #{idx} — MARD: {error_pct:.0f}%',
                     color='#ef4444', fontweight='bold')
        ax.legend(fontsize=9)
        ax.set_ylabel('Glucose (mg/dL)')

    plt.tight_layout()
    plt.savefig('../results/plots/03_failure_cases.png', bbox_inches='tight')
    plt.show()
else:
    print('No Zone C/D/E predictions — excellent model performance!')

## 6. What I learned

*(Fill this in after running the notebook with a trained model)*

**Observations to investigate:**
- Does MARD degrade more at the 60-min horizon than expected?
  If so, the LSTM may not be capturing long-range trends — consider a larger hidden size or
  extending the input window from 2h → 4h.
- Are most failures during rapid glucose rises (postprandial)?
  Adding meal event flags as auxiliary input features would likely improve performance here.
- Does Zone A percentage stay above 95%?
  Below 95% Zone A is the threshold where the clinical community would flag the model as
  unsafe for advisory use.